In [ ]:
# ==============================================================================
# UNIVERSAL PATH CONFIGURATION - REPRODUCIBILITY PIPELINE
# ==============================================================================
RUNNING_IN_COLAB = True

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # This path will look for the shortcut users save to their own Drive
    # Make sure the shared folder name matches here
    ROOT_PATH = '/content/drive/MyDrive/1wEi3ytfVBpxJdc0TjlEQbOhlbxpgjcqa/'
else:
    # Standard path for local computer execution
    ROOT_PATH = './confocal_data/'

# Example global variables (adjust these to your actual file names)
INPUT_LIF_FILE = ROOT_PATH + "3S.lif"

# Pipeline MSSR — análisis consolidado (5 sesiones)

Notebook de ejecución multi-sesión. Procesa cada carpeta de adquisición en su propio `BASE_DIR`, fusiona los `features.csv` en uno consolidado y dispara el modelo mixto (stage 4) sobre el conjunto completo.

**Orden de uso (al inicio de cada sesión de Colab):**
1. Celda 1: instalar dependencias (Colab las pierde entre sesiones).
2. Celda 2: montar Google Drive.
3. Celda 3: cargar el paquete con recarga limpia.
4. Celda 4: rellenar la configuración de sesiones (solo la primera vez).
5. Celda 5: validar antes de empezar (recomendado).
6. Celda 6: ejecutar el pipeline completo.
7. Celda 7+: inspeccionar resultados.

Toda la lógica vive en el paquete `MSSR_PIPELINE/`. Si necesitas ajustar parámetros, edita `config.py` o `run_full_dataset.py`.

## 1. Dependencias

In [ ]:
!pip install oiffile tifffile scikit-image opencv-python-headless scipy statsmodels seaborn --quiet
print("Dependencias instaladas.")

Dependencias instaladas.


## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Cargar el paquete con recarga limpia
Borra `__pycache__` y descarga módulos viejos, asegurando que cualquier edición que hayas hecho en `MSSR_PIPELINE/` se refleje. Si moviste el paquete, ajusta `PKG`.

In [ ]:
import sys, shutil, os

PKG = "/content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/MSSR_PIPELINE"

shutil.rmtree(os.path.join(PKG, "__pycache__"), ignore_errors=True)
for m in ["config", "mssr_core",
          "stage0_convert", "stage1_segmentation", "stage2_mssr",
          "stage3_features", "stage4_regression",
          "run_full_dataset"]:
    sys.modules.pop(m, None)
if PKG not in sys.path:
    sys.path.insert(0, PKG)

import config as cfg
import run_full_dataset as rfd
print("Paquete cargado desde:", PKG)
print("Stages disponibles  :", [m for m in dir(rfd) if m.startswith('run')])

Paquete cargado desde: /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/MSSR_PIPELINE
Stages disponibles  : ['run']


## 4. Configuración de las 5 sesiones

Edita `rfd.SESSIONS` con las rutas reales (o, equivalentemente, edita el archivo `run_full_dataset.py` directamente). Cada entrada tiene:

- `name`        — etiqueta corta que aparecerá en `features.csv` (col `session`)
- `raw_oib_dir` — carpeta plana con los `.oib` crudos
- `base_dir`    — donde irán los TIFF, máscaras, MSSR y `samples.csv` de esa sesión

Las sesiones que todavía tengan placeholders `<TODO>` se omiten automáticamente sin error, así que puedes ir agregando carpetas conforme las tengas listas. La sesión `260327` ya viene precompletada con las rutas que estabas usando.

In [ ]:
# Edita esto y vuelve a correr la celda cada vez que añadas una sesión.
ROOT = "/content/drive/MyDrive/2025 Josue Villegas/LNMA 2026"

rfd.SESSIONS = [
    {"name": "260327",
     "raw_oib_dir": f"{ROOT}/260327/260327",
     "base_dir":    f"{ROOT}/260327/Analisis_MSSR"},

    {"name": "260326",
     "raw_oib_dir": f"{ROOT}/260326/260326",
     "base_dir":    f"{ROOT}/260326/Analisis_MSSR"},

    {"name": "260325",
     "raw_oib_dir": f"{ROOT}/260325/260325",
     "base_dir":    f"{ROOT}/260325/Analisis_MSSR"},

    {"name": "260324",
     "raw_oib_dir": f"{ROOT}/260324/260324",
     "base_dir":    f"{ROOT}/260324/Analisis_MSSR"},

    {"name": "260323",
     "raw_oib_dir": f"{ROOT}/260323/260323",
     "base_dir":    f"{ROOT}/260323/Analisis_MSSR"},
]

rfd.FULL_BASE_DIR      = f"{ROOT}/full_analysis"
rfd.SHARED_BOTTLES_CSV = f"{rfd.FULL_BASE_DIR}/bottles.csv"
rfd.SHARED_INT_CSV     = f"{rfd.FULL_BASE_DIR}/int_samples.csv"

print("Sesiones configuradas:")
for s in rfd.SESSIONS:
    placeholder = "⚠ pendiente" if "<TODO" in s['name'] or "<TODO" in s['raw_oib_dir'] else "✓ lista"
    print(f"  {placeholder}  {s['name']}")
print(f"\nConsolidado en: {rfd.FULL_BASE_DIR}")

Sesiones configuradas:
  ✓ lista  260327
  ✓ lista  260326
  ✓ lista  260325
  ✓ lista  260324
  ✓ lista  260323

Consolidado en: /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/full_analysis


## 5. Validación previa
Antes de procesar, verifica que existen las carpetas de adquisición y que `bottles.csv` e `int_samples.csv` están en `FULL_BASE_DIR/`. Si la celda imprime problemas, créalos/cópialos antes de seguir.

In [ ]:
# Si full_analysis/ aún no existe, créalo y copia las dos CSV compartidas.
os.makedirs(rfd.FULL_BASE_DIR, exist_ok=True)

import pandas as pd
for name, path in [("bottles.csv", rfd.SHARED_BOTTLES_CSV),
                   ("int_samples.csv", rfd.SHARED_INT_CSV)]:
    if os.path.exists(path):
        print(f"  ✓ {name}: existe")
        print(pd.read_csv(path).head(3).to_string(index=False))
    else:
        print(f"  ✗ {name}: NO existe en {path}")

print("\nRevisión de carpetas de adquisición:")
for s in rfd.SESSIONS:
    if "<TODO" in s['raw_oib_dir']:
        continue
    if os.path.exists(s['raw_oib_dir']):
        n = sum(1 for f in os.listdir(s['raw_oib_dir']) if f.lower().endswith('.oib'))
        print(f"  ✓ {s['name']}: {n} archivos .oib en {s['raw_oib_dir']}")
    else:
        print(f"  ✗ {s['name']}: NO existe {s['raw_oib_dir']}")

  ✓ bottles.csv: existe
 bottle  temperature_C  respiration_uM_O2_h  is_control                                              notes
      1           16.0                 2.69       False                                   Origen de INT 10
      4           17.1                 1.44       False Origen de INT 13 e imaginada directamente como BOD
     12           16.0                 2.41       False                                Origen de INT 7 y 8
  ✓ int_samples.csv: existe
 int_sample  bottle_bod  notes
          1          21    NaN
          2          21    NaN
          3          24    NaN

Revisión de carpetas de adquisición:
  ✓ 260327: 100 archivos .oib en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260327/260327
  ✓ 260326: 142 archivos .oib en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260326/260326
  ✓ 260325: 151 archivos .oib en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260325/260325
  ✓ 260324: 85 archivos .oib en /content/drive/MyDrive/202

## 6. Ejecutar el pipeline completo

Llama una sola vez y corre todo: stages 0–3 por sesión, fusión de `features.csv`, stage 4 sobre el consolidado.

**Opciones útiles:**
- `rfd.run(skip_if_features_exist=True)`: omite las sesiones cuyo `features.csv` ya existe. Útil cuando vas agregando carpetas y no quieres reprocesar la piloto.
- `rfd.run(stages=("3",))`: re-extrae features sin reprocesar máscaras o MSSR (si solo cambiaste `stage3_features.py`).
- `rfd.run(run_stage4=False)`: corre todo menos el modelo estadístico (útil para revisar QC primero).

La primera ejecución sobre las 5 sesiones tarda del orden de **decenas de minutos** (MSSR es el cuello de botella). Reejecuciones parciales son rápidas.

In [ ]:
rfd.run(skip_if_features_exist=True)

Cargando paquete MSSR…

Sesiones a procesar: ['260327', '260326', '260325', '260324', '260323']
Stages por sesión:   ['0', '1', '2', '3']
Stage 4 al final:    True


SESIÓN: 260327
  features.csv ya existe en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260327/Analisis_MSSR/features.csv. Salto (skip_if_features_exist=True).

SESIÓN: 260326
  features.csv ya existe en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260326/Analisis_MSSR/features.csv. Salto (skip_if_features_exist=True).

SESIÓN: 260325
  features.csv ya existe en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260325/Analisis_MSSR/features.csv. Salto (skip_if_features_exist=True).

SESIÓN: 260324
  features.csv ya existe en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260324/Analisis_MSSR/features.csv. Salto (skip_if_features_exist=True).

SESIÓN: 260323
  features.csv ya existe en /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/260323/Analisis_MSSR/features.csv. Salto (skip_if_features_exi

## 7. Inspección del consolidado
Carga el `features.csv` fusionado y revisa cobertura por botella, sesión y condición.

In [ ]:
import pandas as pd

features = pd.read_csv(f"{rfd.FULL_BASE_DIR}/features.csv")
print(f"Total: {len(features):,} células × {features.shape[1]} columnas")
print(f"Sesiones    : {sorted(features['session'].unique())}")
print(f"Botellas    : {sorted(features['bottle'].unique())}")
print(f"Tratamientos: {sorted(features['treatment'].unique())}")
print(f"Fluorocromos: {sorted(features['fluorophore'].unique())}")
print(f"Tasas O₂    : {sorted(features['respiration_uM_O2_h'].unique())}")

print("\nCélulas por sesión × botella × tratamiento × fluorocromo:")
pivot = (features.groupby(['session','bottle','treatment','fluorophore'])
         .size().unstack(['treatment','fluorophore'], fill_value=0))
print(pivot)

Total: 1,507 células × 49 columnas
Sesiones    : [np.int64(260323), np.int64(260324), np.int64(260325), np.int64(260326), np.int64(260327)]
Botellas    : [np.int64(4), np.int64(12), np.int64(13), np.int64(19), np.int64(20), np.int64(21), np.int64(24)]
Tratamientos: ['BOD', 'INT']
Fluorocromos: ['DAPI', 'SYBR']
Tasas O₂    : [np.float64(0.39), np.float64(0.47), np.float64(0.54), np.float64(1.28), np.float64(1.44), np.float64(2.11), np.float64(2.41)]

Células por sesión × botella × tratamiento × fluorocromo:
treatment       BOD       INT     
fluorophore    DAPI SYBR DAPI SYBR
session bottle                    
260323  4        26    0    0    0
        12       44   74   39   69
        13        0   33    0    0
        19        0    0   49    0
260324  13       52    0   38   41
        20       24   58    0   54
260325  4        26   10   37   45
        19        0   25   44   72
        20        0    0   30    0
260326  12        0    0    0   49
        20        0    0    0   6

## 8. Resultados del modelo mixto (stage 4)
Carga `feature_results.csv` y muestra las features ordenadas por evidencia de **interacción tratamiento × respiración** (la hipótesis central).

In [ ]:
results = pd.read_csv(f"{rfd.FULL_BASE_DIR}/STATS/feature_results.csv")

top = (results.dropna(subset=['q_interaction'])
       .sort_values('q_interaction'))

print("Top features por interacción tratamiento × respiración (q-valor FDR):")
print(top[['feature','fluorophore','beta_treatment','beta_interaction',
          'p_interaction','q_interaction']].head(15).to_string(index=False))

Top features por interacción tratamiento × respiración (q-valor FDR):
            feature fluorophore  beta_treatment  beta_interaction  p_interaction  q_interaction
mssr_n_local_minima        DAPI       -9.023651          3.667695       0.000039       0.001430
            raw_sum        DAPI  -365669.235258     116099.564980       0.000115       0.002119
            session        SYBR       -0.140512          0.379522       0.000264       0.004302
            raw_min        SYBR      174.156518         81.939574       0.000340       0.004302
         raw_median        SYBR      290.414417        171.144214       0.000339       0.004302
           raw_mean        DAPI    -1051.465060        244.852245       0.000590       0.005454
            raw_min        DAPI     -564.593329        115.319933       0.000479       0.005454
         raw_median        DAPI    -1041.227097        236.040177       0.000941       0.006612
           mssr_sum        DAPI   -55220.967468      16277.239976 

---
## Utilidades

### Limpieza segura entre corridas
Borra máscaras, MSSR, features y stats sin tocar los `.tif` de entrada de `RAW/`. Útil al cambiar parámetros de segmentación.

In [ ]:
for s in rfd.SESSIONS:
    if "<TODO" in s['base_dir']:
        continue
    for sub in ("INT/MASKS", "BOD/MASKS",
                "INT/MSSR",  "BOD/MSSR",
                "STATS"):
        shutil.rmtree(os.path.join(s['base_dir'], sub), ignore_errors=True)
    for f in ("features.csv",):
        p = os.path.join(s['base_dir'], f)
        if os.path.exists(p): os.remove(p)
shutil.rmtree(os.path.join(rfd.FULL_BASE_DIR, "STATS"), ignore_errors=True)
for f in ("features.csv",):
    p = os.path.join(rfd.FULL_BASE_DIR, f)
    if os.path.exists(p): os.remove(p)
print("Limpieza hecha; los .tif crudos de cada RAW/ se conservan.")

### Re-ejecutar una sola sesión (debug)

In [ ]:
# Por ejemplo, re-correr solo stage 3 para 260327:
rfd.run(sessions=[rfd.SESSIONS[0]], stages=("3",), run_stage4=False)

In [ ]:
import sys, shutil, os
PKG = "/content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/MSSR_PIPELINE"
shutil.rmtree(f"{PKG}/__pycache__", ignore_errors=True)
sys.modules.pop("stage4_regression", None)
sys.modules.pop("config", None)
sys.modules.pop("run_full_dataset", None)

import config as cfg, run_full_dataset as rfd, stage4_regression as s4

# Hay que apuntar cfg.BASE_DIR al consolidado, porque por defecto está
# apuntando a la última sesión que el driver procesó.
cfg.BASE_DIR = rfd.FULL_BASE_DIR
print("Trabajando sobre:", cfg.BASE_DIR)

results = s4.run_all(top_k_panels=8)

Trabajando sobre: /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/full_analysis
features.csv: 1507 células × 49 columnas

=== DAPI ===
=== SYBR ===

feature_results.csv → /content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/full_analysis/STATS/feature_results.csv

Top features por interacción tratamiento × respiración:
            feature fluorophore  beta_interaction  p_interaction  q_interaction
mssr_n_local_minima        DAPI          3.667695       0.000039       0.001391
            raw_sum        DAPI     116099.564980       0.000115       0.002061
            raw_min        DAPI        115.319933       0.000479       0.005307
           raw_mean        DAPI        244.852245       0.000590       0.005307
         raw_median        SYBR        171.144214       0.000339       0.006283
            raw_min        SYBR         81.939574       0.000340       0.006283
           mssr_sum        DAPI      16277.239976       0.001072       0.006433
         raw_median        DAPI   